# Understanding Our Dataset

This notebook walks through every piece of data we use — where it comes from, what it looks like,
why we chose it, and how it feeds into the DQN pipeline.

**Structure:**
1. Raw price data — what are we actually trading?
2. Macro context data — what tells us about the regime?
3. Spread construction — how do we build the signal?
4. OU parameter estimation — how do we model mean-reversion?
5. Kelly criterion — how do we size positions?
6. Full feature matrix — what does the DQN actually see?
7. Train/test split — temporal integrity
8. The problem: structural traps

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load everything from the pipeline
spreads_df = pd.read_parquet('v0/datasets/spreads.parquet')
df_prices = pd.read_parquet('v0/datasets/pair_prices.parquet')
df_macro = pd.read_parquet('v0/datasets/macro.parquet')

with open('v0/datasets/nb02_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)

cointegrated_pairs = artifacts['cointegrated_pairs']
STATIC_FEATURE_COLS = artifacts['STATIC_FEATURE_COLS']
scaler = artifacts['scaler']
pca = artifacts['pca']

all_features = {}
all_ou_params = {}
for pair_name in cointegrated_pairs:
    safe = pair_name.replace('/', '_')
    all_features[pair_name] = pd.read_parquet(f'v0/datasets/features_{safe}.parquet')
    all_ou_params[pair_name] = pd.read_parquet(f'v0/datasets/ou_params_{safe}.parquet')

TRAIN_END = '2019-12-31'
TEST_START = '2020-01-01'

print(f'Loaded {len(cointegrated_pairs)} cointegrated pairs')
print(f'Price data: {df_prices.shape}')
print(f'Macro data: {df_macro.shape}')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/li

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/li

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/elen4904/li

AttributeError: _ARRAY_API not found

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

---
## 1. Raw Price Data

Daily closing prices from yfinance for 46 NYSE tickers + VIX, from 2010-01-01 to 2023-12-31.
These are adjusted close prices (splits/dividends handled by yfinance).

In [ ]:
print('=== Price Data Overview ===')
print(f'Shape: {df_prices.shape[0]} trading days x {df_prices.shape[1]} tickers')
print(f'Date range: {df_prices.index[0].date()} to {df_prices.index[-1].date()}')
print(f'\nTickers: {list(df_prices.columns)}')
print(f'\nMissing values per ticker:')
missing = df_prices.isna().sum()
print(missing[missing > 0] if missing.any() else 'None')
print(f'\nBasic stats (first 5 tickers):')
df_prices.iloc[:, :5].describe().round(2)

In [ ]:
# What do the prices actually look like?
# Show one pair from each sector to understand the relationships
demo_pairs = [('KO', 'PEP', 'Consumer Staples'), 
              ('XOM', 'CVX', 'Energy'),
              ('JPM', 'GS', 'Finance'),
              ('AAPL', 'MSFT', 'Tech'),
              ('JNJ', 'PFE', 'Healthcare')]

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
for idx, (a, b, sector) in enumerate(demo_pairs):
    ax = axes[idx]
    # Normalize to 100 at start for comparison
    pa = df_prices[a].dropna()
    pb = df_prices[b].dropna()
    ax.plot(pa.index, pa / pa.iloc[0] * 100, label=a, linewidth=1)
    ax.plot(pb.index, pb / pb.iloc[0] * 100, label=b, linewidth=1)
    ax.axvline(pd.Timestamp(TEST_START), color='red', linestyle='--', alpha=0.5, label='Train/Test split')
    ax.set_ylabel('Normalized Price')
    ax.set_title(f'{sector}: {a} vs {b}', fontweight='bold', loc='left')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Normalized Prices — Do These Pairs Move Together?', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key observation: pairs that move together can diverge temporarily (tradeable),')
print('but sometimes they diverge permanently (structural trap).')

In [ ]:
# Daily returns distribution — are these assets behaving normally?
returns = df_prices.pct_change().dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Return distributions for a few tickers
for ax, ticker in zip(axes.flat, ['KO', 'XOM', 'JPM', 'AAPL']):
    ret = returns[ticker]
    ax.hist(ret, bins=100, density=True, alpha=0.7, color='steelblue', edgecolor='none')
    ax.axvline(ret.mean(), color='red', linestyle='--', label=f'mean={ret.mean():.4f}')
    ax.set_title(f'{ticker} daily returns', fontweight='bold')
    ax.set_xlabel('Daily return')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    # Print fat tail stats
    print(f'{ticker}: mean={ret.mean():.5f}, std={ret.std():.4f}, '
          f'skew={ret.skew():.2f}, kurtosis={ret.kurtosis():.2f}')

fig.suptitle('Daily Return Distributions — Fat Tails Everywhere', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of all tickers — which pairs are most correlated?
corr = returns.drop(columns=['VIX'], errors='ignore').corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-0.5, vmax=1,
            square=True, linewidths=0.5, ax=ax, fmt='.1f',
            annot=True, annot_kws={'size': 6})
ax.set_title('Return Correlations Across Universe (2010-2023)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print('High correlation = candidate for pairs trading.')
print('But correlation != cointegration. A pair can be correlated but not mean-reverting.')

---
## 2. Macro Context Data

From FRED (Federal Reserve Economic Data):
- **10Y Treasury Yield** (`DGS10`) — rate environment, affects all assets
- **High-Yield Credit Spread** (`BAMLC0A0CM`) — ICE BofA IG/HY spread, measures credit stress

From yfinance:
- **VIX** (`^VIX`) — implied volatility of S&P 500, the "fear gauge"

These are the macro regime indicators that tell the DQN whether market conditions support mean-reversion.

In [ ]:
print('=== Macro Data Overview ===')
print(f'Shape: {df_macro.shape}')
print(f'Columns: {list(df_macro.columns)}')
print(f'\nDate range: {df_macro.index[0].date()} to {df_macro.index[-1].date()}')
print(f'\nSummary stats:')
display(df_macro.describe().round(2))

print(f'\nVIX summary (from price data):')
vix = df_prices['VIX'].dropna()
print(f'  Mean: {vix.mean():.1f}, Median: {vix.median():.1f}')
print(f'  Min: {vix.min():.1f}, Max: {vix.max():.1f}')
print(f'  Days VIX > 25: {(vix > 25).sum()} ({(vix > 25).mean()*100:.1f}%)')
print(f'  Days VIX > 30: {(vix > 30).sum()} ({(vix > 30).mean()*100:.1f}%)')

In [ ]:
# Visualize macro regimes over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# VIX
ax = axes[0]
ax.plot(vix.index, vix, color='#2c3e50', linewidth=0.8)
ax.axhline(20, color='green', linestyle='--', alpha=0.5, label='Calm (<20)')
ax.axhline(30, color='red', linestyle='--', alpha=0.5, label='Stress (>30)')
ax.fill_between(vix.index, 0, vix, where=vix > 30, color='red', alpha=0.2)
ax.set_ylabel('VIX')
ax.set_title('VIX — Market Volatility', fontweight='bold', loc='left')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# 10Y Yield
ax = axes[1]
ax.plot(df_macro.index, df_macro['10Y_Yield'], color='#e67e22', linewidth=0.8)
ax.set_ylabel('Yield (%)')
ax.set_title('10-Year Treasury Yield — Rate Regime', fontweight='bold', loc='left')
ax.grid(True, alpha=0.3)

# HY Spread
ax = axes[2]
ax.plot(df_macro.index, df_macro['HY_Spread'], color='#e74c3c', linewidth=0.8)
ax.set_ylabel('Spread (%)')
ax.set_title('High-Yield Credit Spread — Credit Stress', fontweight='bold', loc='left')
ax.grid(True, alpha=0.3)

for ax in axes:
    ax.axvline(pd.Timestamp(TEST_START), color='black', linestyle='--', alpha=0.5)
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-06-01'), color='red', alpha=0.08)
    ax.axvspan(pd.Timestamp('2022-03-01'), pd.Timestamp('2023-01-01'), color='orange', alpha=0.08)

axes[0].annotate('COVID', xy=(pd.Timestamp('2020-03-15'), 70), fontsize=9, color='red')
axes[0].annotate('Rate Hikes', xy=(pd.Timestamp('2022-06-01'), 35), fontsize=9, color='orange')
axes[0].annotate('TRAIN', xy=(pd.Timestamp('2015-01-01'), 5), fontsize=10, fontweight='bold')
axes[0].annotate('TEST', xy=(pd.Timestamp('2021-06-01'), 5), fontsize=10, fontweight='bold', color='red')

fig.suptitle('Macro Regime Context — What the DQN Uses to Detect Danger', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key: the test period (2020-2023) contains two extreme macro events.')
print('This is why static thresholds fail — they cannot see these regime shifts.')

In [ ]:
# How correlated are the macro indicators with each other?
macro_combined = df_prices[['VIX']].join(df_macro).dropna()
macro_combined['VIX_5d_chg'] = macro_combined['VIX'].pct_change(5)

print('Macro indicator correlations:')
print(macro_combined.corr().round(3))

# Scatter: VIX vs HY spread — do they spike together?
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(macro_combined['VIX'], macro_combined['HY_Spread'], 
           alpha=0.2, s=5, color='#2c3e50')
ax.set_xlabel('VIX')
ax.set_ylabel('HY Credit Spread (%)')
ax.set_title('VIX vs Credit Spread — Stress Indicators Move Together', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Spread Construction

For each pair (A, B), the spread is:

$$S_t = B_t - \beta_t \cdot A_t$$

where $\beta_t$ is a **rolling 252-day OLS hedge ratio** (no look-ahead). This is the signal we trade on —
when the spread deviates from its mean, we bet on reversion.

In [ ]:
print('=== Spread Data Overview ===')
print(f'Shape: {spreads_df.shape}')
print(f'Pairs: {list(spreads_df.columns)}')
print(f'\nObservations per pair:')
for col in spreads_df.columns:
    n = spreads_df[col].dropna().shape[0]
    print(f'  {col:12s}: {n} days')

print(f'\nSpread summary stats:')
display(spreads_df.describe().round(2))

In [ ]:
# Show what the spreads look like — are they mean-reverting?
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

example_pairs = ['KO/PEP', 'XOM/CVX', 'JPM/GS', 'AAPL/MSFT']

for idx, pair in enumerate(example_pairs):
    ax = axes[idx]
    spread = spreads_df[pair].dropna()
    roll_mean = spread.rolling(60).mean()
    roll_std = spread.rolling(60).std()
    
    ax.plot(spread.index, spread, color='#2c3e50', linewidth=0.7, alpha=0.8, label='Spread')
    ax.plot(roll_mean.index, roll_mean, color='red', linestyle='--', linewidth=1, label='60d mean')
    ax.fill_between(roll_mean.index, roll_mean - roll_std, roll_mean + roll_std,
                    color='red', alpha=0.1, label='$\pm 1\sigma$')
    ax.axvline(pd.Timestamp(TEST_START), color='black', linestyle='--', alpha=0.5)
    ax.set_title(pair, fontweight='bold', loc='left')
    ax.set_ylabel('Spread')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Raw Spreads with Rolling Mean/Std — Are They Mean-Reverting?',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Notice: spreads oscillate around the mean (good for trading),')
print('but the mean itself drifts over time (why we use rolling windows).')

In [ ]:
# Z-scores — the actual trading signal
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for idx, pair in enumerate(example_pairs):
    ax = axes[idx]
    spread = spreads_df[pair].dropna()
    roll_mean = spread.rolling(60).mean()
    roll_std = spread.rolling(60).std()
    z = ((spread - roll_mean) / roll_std.replace(0, np.nan)).dropna()
    
    ax.plot(z.index, z, color='#2c3e50', linewidth=0.6)
    ax.axhline(1.0, color='red', linestyle='--', alpha=0.6, label='Short entry (+1$\sigma$)')
    ax.axhline(-1.0, color='green', linestyle='--', alpha=0.6, label='Long entry (-1$\sigma$)')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(pd.Timestamp(TEST_START), color='black', linestyle='--', alpha=0.5)
    ax.fill_between(z.index, -1, 1, color='gray', alpha=0.05)
    ax.set_title(pair, fontweight='bold', loc='left')
    ax.set_ylabel('Z-score')
    ax.set_ylim(-4, 4)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Z-Scores — The Static Trading Signal\n'
             'Every time Z crosses $\pm$1, a static strategy enters a trade. But should it?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Ornstein-Uhlenbeck Parameters

We model each spread as an OU process: $dS_t = \theta(\mu - S_t)dt + \sigma dW_t$

Estimated via AR(1) regression on rolling 60-day windows:
- **$\theta$** — mean-reversion speed (higher = faster reversion = safer to trade)
- **$\mu$** — equilibrium level
- **$\sigma$** — volatility of the process
- **half-life** = $\ln(2)/\theta$ — days until halfway back to mean

In [ ]:
# OU parameter summary across all pairs
ou_summary = []
for pair_name in cointegrated_pairs:
    ou = all_ou_params[pair_name]
    stationary = ou[ou['is_stationary']]
    ou_summary.append({
        'Pair': pair_name,
        'Stationary %': ou['is_stationary'].mean() * 100,
        'Median theta': stationary['theta'].median(),
        'Median half-life (days)': stationary['half_life'].median(),
        'Median sigma': stationary['sigma'].median(),
        'Theta std': stationary['theta'].std(),
    })

ou_summary_df = pd.DataFrame(ou_summary).set_index('Pair')
print('=== OU Parameter Summary (Rolling 60-day Windows) ===')
display(ou_summary_df.round(2))

print(f'\nAverage stationarity rate: {ou_summary_df["Stationary %"].mean():.1f}%')
print('This means ~30% of the time, the spread is NOT stationary — trading is dangerous.')

In [ ]:
# Show theta over time — the key instability
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for idx, pair in enumerate(example_pairs):
    ax = axes[idx]
    ou = all_ou_params[pair]
    theta = ou['theta'].copy()
    theta[~ou['is_stationary']] = np.nan
    
    ax.plot(theta.index, theta, color='#2c3e50', linewidth=0.7)
    ax.axhline(theta.median(), color='red', linestyle='--', alpha=0.5, 
               label=f'median={theta.median():.1f}')
    ax.axvline(pd.Timestamp(TEST_START), color='black', linestyle='--', alpha=0.5)
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-06-01'), color='red', alpha=0.1)
    ax.axvspan(pd.Timestamp('2022-03-01'), pd.Timestamp('2023-01-01'), color='orange', alpha=0.1)
    ax.set_title(pair, fontweight='bold', loc='left')
    ax.set_ylabel('$\\theta$')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Rolling $\\theta$ (Mean-Reversion Speed) — The Core Instability\n'
             'When $\\theta$ drops or goes non-stationary, static thresholds fail.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of theta and half-life across all pairs and times
all_thetas = []
all_halflives = []
for pair in cointegrated_pairs:
    ou = all_ou_params[pair]
    stat = ou[ou['is_stationary']]
    all_thetas.extend(stat['theta'].values)
    all_halflives.extend(stat['half_life'].values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(all_thetas, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax1.axvline(np.median(all_thetas), color='red', linestyle='--', label=f'median={np.median(all_thetas):.1f}')
ax1.set_xlabel('$\\theta$ (mean-reversion speed)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of $\\theta$ Across All Pairs/Windows', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.hist(all_halflives, bins=80, color='coral', edgecolor='none', alpha=0.8)
ax2.axvline(np.median(all_halflives), color='red', linestyle='--', label=f'median={np.median(all_halflives):.0f}d')
ax2.set_xlabel('Half-life (trading days)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Half-Life', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Theta: mean={np.mean(all_thetas):.2f}, median={np.median(all_thetas):.2f}, std={np.std(all_thetas):.2f}')
print(f'Half-life: mean={np.mean(all_halflives):.0f}d, median={np.median(all_halflives):.0f}d')
print(f'\nThis wide variation is WHY we need adaptive thresholds — theta is not stable.')

---
## 5. Kelly Criterion

Position sizing from the OU transition density (Lecture 9, slides 36-39):

$$f^* = \frac{pK - q}{K}$$

where $p$ = probability of reversion, $K$ = win/loss ratio, derived from the OU expected value and variance at a 20-day horizon.

In [ ]:
# Kelly fractions across the dataset
all_kelly = []
for pair in cointegrated_pairs:
    feat = all_features[pair]
    all_kelly.extend(feat['kelly_fraction'].values)

all_kelly = np.array(all_kelly)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(all_kelly, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
ax1.set_xlabel('Kelly Fraction $f^*$')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Kelly Fractions', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Kelly fraction vs z-score — does Kelly say to bet more when deviation is large?
kellys = []
zscores = []
for pair in cointegrated_pairs[:5]:
    feat = all_features[pair]
    kellys.extend(feat['kelly_fraction'].values)
    zscores.extend(feat['z_score'].values)

ax2.scatter(zscores, kellys, alpha=0.05, s=3, color='#2c3e50')
ax2.set_xlabel('Z-score')
ax2.set_ylabel('Kelly Fraction')
ax2.set_title('Kelly Fraction vs Z-Score', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Kelly fraction: mean={all_kelly.mean():.4f}, median={np.median(all_kelly):.4f}')
print(f'Zero Kelly (no edge): {(all_kelly == 0).sum()} / {len(all_kelly)} ({(all_kelly == 0).mean()*100:.1f}%)')
print(f'\nWhen Kelly = 0, the OU model says there is no edge — don\'t trade.')
print('The DQN learns to respect this, but also uses macro context on top.')

---
## 6. Full Feature Matrix — What the DQN Sees

14 raw features per timestep, organized into modalities:

| Modality | Features | Purpose |
|---|---|---|
| Cluster stability (6) | theta, mu, sigma, z_score, half_life, ou_resid_var | How strong is mean-reversion right now? |
| Market context (6) | VIX, VIX_5d_change, 10Y yield, HY spread, pair_corr_20d, spread_vol_20d | What macro regime are we in? |
| Kelly (2) | kelly_fraction, kelly_edge | What does the OU model say about edge? |

These 14 features get StandardScaled then PCA-compressed to 11 components.
3 position features (position, unrealized PnL, days held) are appended at runtime.

**Total DQN state dimension: 14**

In [ ]:
# Show the feature matrix for one pair
demo_pair = 'KO/PEP'
feat = all_features[demo_pair]

print(f'=== Feature Matrix for {demo_pair} ===')
print(f'Shape: {feat.shape}')
print(f'Columns: {list(feat.columns)}')
print(f'Date range: {feat.index[0].date()} to {feat.index[-1].date()}')
print(f'\nFirst 3 rows:')
display(feat.head(3))
print(f'\nDescriptive stats:')
display(feat.describe().round(4))

In [ ]:
# Feature correlations — are features redundant?
fig, ax = plt.subplots(figsize=(10, 8))
feat_corr = feat[STATIC_FEATURE_COLS].corr()
sns.heatmap(feat_corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title(f'Feature Correlations — {demo_pair}\n'
             'High correlation = redundancy (why PCA helps)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Notable correlations:')
print('  theta <-> half_life: perfectly inverse (by construction)')
print('  VIX <-> HY spread: both measure stress')
print('  sigma <-> ou_resid_var: related (OU vol vs spread vol)')

In [ ]:
# All 14 features over time for one pair — what does the DQN input look like?
fig, axes = plt.subplots(7, 2, figsize=(16, 20), sharex=True)
axes = axes.flatten()

for idx, col in enumerate(STATIC_FEATURE_COLS):
    ax = axes[idx]
    vals = feat[col].dropna()
    ax.plot(vals.index, vals, color='#2c3e50', linewidth=0.5)
    ax.axvline(pd.Timestamp(TEST_START), color='red', linestyle='--', alpha=0.5)
    ax.set_title(col, fontweight='bold', fontsize=10, loc='left')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'All 14 Raw Features Over Time — {demo_pair}\n'
             'Red line = train/test split', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# PCA — what happens after compression?
print(f'=== PCA Compression ===')
print(f'Input: {len(STATIC_FEATURE_COLS)} features')
print(f'Output: {pca.n_components_} principal components')
print(f'Variance retained: {pca.explained_variance_ratio_.sum():.1%}')
print(f'\nVariance per component:')
for i, var in enumerate(pca.explained_variance_ratio_):
    cumvar = pca.explained_variance_ratio_[:i+1].sum()
    bar = '#' * int(var * 100)
    print(f'  PC{i+1:2d}: {var:6.1%} (cumulative: {cumvar:5.1%}) {bar}')

# PCA loadings — what do the components mean?
fig, ax = plt.subplots(figsize=(12, 6))
loadings = pd.DataFrame(pca.components_[:5].T,
                        index=STATIC_FEATURE_COLS,
                        columns=[f'PC{i+1}' for i in range(5)])
sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('PCA Loadings — What Each Component Captures', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Train/Test Split — Temporal Integrity

- **Train:** 2010-01-01 to 2019-12-31 (pre-COVID)
- **Test:** 2020-01-01 to 2023-12-31 (COVID + rate hikes)

The test set is strictly forward in time. No data leakage.

In [ ]:
# How much data is in each split?
split_stats = []
for pair in cointegrated_pairs:
    feat = all_features[pair]
    train = feat[feat.index <= TRAIN_END]
    test = feat[feat.index >= TEST_START]
    split_stats.append({
        'Pair': pair,
        'Train days': len(train),
        'Test days': len(test),
        'Train %': len(train) / len(feat) * 100,
    })

split_df = pd.DataFrame(split_stats).set_index('Pair')
display(split_df.round(1))

total_train = split_df['Train days'].sum()
total_test = split_df['Test days'].sum()
print(f'\nTotal: {total_train} train observations, {total_test} test observations')
print(f'Ratio: {total_train / (total_train + total_test) * 100:.0f}% train / '
      f'{total_test / (total_train + total_test) * 100:.0f}% test')

In [ ]:
# Feature distribution shift: train vs test
# This is important — if distributions shift dramatically, the model may struggle
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
check_features = ['theta', 'z_score', 'VIX', 'hy_spread', 'kelly_fraction', 'spread_vol_20d']

for ax, col in zip(axes.flat, check_features):
    train_vals = []
    test_vals = []
    for pair in cointegrated_pairs:
        feat = all_features[pair]
        train_vals.extend(feat.loc[feat.index <= TRAIN_END, col].dropna().values)
        test_vals.extend(feat.loc[feat.index >= TEST_START, col].dropna().values)
    
    ax.hist(train_vals, bins=50, density=True, alpha=0.6, color='steelblue', label='Train')
    ax.hist(test_vals, bins=50, density=True, alpha=0.6, color='coral', label='Test')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Feature Distributions: Train vs Test\n'
             'Shifts here explain why the model faces challenges in the test period',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Look for: VIX distribution shifts right (more stress in test),')
print('HY spread shifts right (credit stress), theta may shift (less stationarity).')

---
## 8. The Problem: Structural Traps

A "structural trap" is when the z-score crosses the threshold (signaling a trade),
but instead of reverting, the spread keeps diverging. The static strategy gets stuck in a losing position.

We measure this by looking at whether the z-score is closer to zero 20 days after a signal.

In [ ]:
# Compute trap rates per pair
trap_results = []

for pair in cointegrated_pairs:
    spread = spreads_df[pair].dropna()
    roll_mean = spread.rolling(60).mean()
    roll_std = spread.rolling(60).std()
    z = ((spread - roll_mean) / roll_std.replace(0, np.nan)).dropna()
    
    signals = 0
    traps = 0
    
    for i in range(len(z) - 20):
        z_now = z.iloc[i]
        z_future = z.iloc[i + 20]
        
        if abs(z_now) > 1.0:  # signal triggered
            signals += 1
            # Trap = spread moved further from zero
            if abs(z_future) > abs(z_now):
                traps += 1
    
    trap_rate = traps / signals if signals > 0 else 0
    trap_results.append({'Pair': pair, 'Signals': signals, 'Traps': traps,
                         'Trap Rate %': trap_rate * 100})

trap_df = pd.DataFrame(trap_results).set_index('Pair')
display(trap_df.round(1))

print(f'\nAverage trap rate: {trap_df["Trap Rate %"].mean():.1f}%')
print(f'Range: {trap_df["Trap Rate %"].min():.1f}% to {trap_df["Trap Rate %"].max():.1f}%')
print(f'\nThis means roughly 1 in 3 trades based on static thresholds FAILS.')
print('This is the problem our DQN is trying to solve.')

In [ ]:
# Visualize: what do traps look like vs good trades?
# Pick one pair, find examples of each
pair = 'KO/PEP'
spread = spreads_df[pair].dropna()
roll_mean = spread.rolling(60).mean()
roll_std = spread.rolling(60).std()
z = ((spread - roll_mean) / roll_std.replace(0, np.nan)).dropna()

# Find trap and reversion examples
trap_examples = []
good_examples = []

for i in range(len(z) - 40):
    z_now = z.iloc[i]
    if abs(z_now) > 1.5:  # strong signal
        z_future = z.iloc[i+1:i+40]
        if abs(z.iloc[i+20]) > abs(z_now) * 1.2:  # clear trap
            trap_examples.append(i)
        elif abs(z.iloc[i+20]) < abs(z_now) * 0.5:  # clear reversion
            good_examples.append(i)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (examples, label, color) in enumerate([
    (good_examples[:3], 'GOOD TRADE (Reverted)', 'green'),
    (trap_examples[:3], 'STRUCTURAL TRAP (Diverged)', 'red')
]):
    for row, ex_idx in enumerate(examples[:3]):
        if row >= 3:
            break
        ax_idx = row if col == 0 else row
        ax = axes[col][row]
        window = z.iloc[max(0,ex_idx-10):ex_idx+40]
        ax.plot(range(len(window)), window.values, color=color, linewidth=1.5)
        ax.axvline(10, color='black', linestyle='--', alpha=0.5, label='Signal')
        ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
        ax.axhline(-1.0, color='gray', linestyle=':', alpha=0.5)
        ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
        ax.set_title(f'{label}\n{window.index[10].date()}', fontsize=9, fontweight='bold')
        ax.set_xlabel('Days from signal')
        ax.set_ylabel('Z-score')
        ax.grid(True, alpha=0.3)

fig.suptitle(f'{pair}: Good Trades vs Structural Traps\n'
             'Top = reverted back to 0. Bottom = diverged further.',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Do traps cluster in high-VIX environments?
trap_vix = []
good_vix = []

for pair in cointegrated_pairs:
    spread = spreads_df[pair].dropna()
    feat = all_features[pair]
    roll_mean = spread.rolling(60).mean()
    roll_std = spread.rolling(60).std()
    z = ((spread - roll_mean) / roll_std.replace(0, np.nan)).dropna()
    
    common = z.index.intersection(feat.index)
    
    for i in range(len(common) - 20):
        date = common[i]
        z_now = z.loc[date]
        
        if abs(z_now) > 1.0 and date in feat.index:
            future_date = common[min(i+20, len(common)-1)]
            z_future = z.loc[future_date]
            vix_val = feat.loc[date, 'VIX']
            hy_val = feat.loc[date, 'hy_spread']
            theta_val = feat.loc[date, 'theta']
            
            if abs(z_future) > abs(z_now):  # trap
                trap_vix.append({'VIX': vix_val, 'HY_Spread': hy_val, 'Theta': theta_val})
            else:  # good
                good_vix.append({'VIX': vix_val, 'HY_Spread': hy_val, 'Theta': theta_val})

trap_df_vix = pd.DataFrame(trap_vix)
good_df_vix = pd.DataFrame(good_vix)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# VIX distribution
axes[0].hist(good_df_vix['VIX'], bins=40, density=True, alpha=0.6, color='green', label='Good trades')
axes[0].hist(trap_df_vix['VIX'], bins=40, density=True, alpha=0.6, color='red', label='Traps')
axes[0].set_xlabel('VIX at signal time')
axes[0].set_title('VIX Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# HY spread distribution
axes[1].hist(good_df_vix['HY_Spread'], bins=40, density=True, alpha=0.6, color='green', label='Good trades')
axes[1].hist(trap_df_vix['HY_Spread'], bins=40, density=True, alpha=0.6, color='red', label='Traps')
axes[1].set_xlabel('HY Spread at signal time')
axes[1].set_title('Credit Spread Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Theta distribution
axes[2].hist(good_df_vix['Theta'], bins=40, density=True, alpha=0.6, color='green', label='Good trades')
axes[2].hist(trap_df_vix['Theta'], bins=40, density=True, alpha=0.6, color='red', label='Traps')
axes[2].set_xlabel('$\\theta$ at signal time')
axes[2].set_title('Mean-Reversion Speed Distribution', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle('Macro/OU Context When Traps Happen vs Good Trades\n'
             'If distributions differ, these features help predict traps.',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print(f'Good trades: mean VIX={good_df_vix["VIX"].mean():.1f}, mean theta={good_df_vix["Theta"].mean():.2f}')
print(f'Traps:       mean VIX={trap_df_vix["VIX"].mean():.1f}, mean theta={trap_df_vix["Theta"].mean():.2f}')
print(f'\nIf traps have higher VIX and lower theta, the DQN has learnable signal.')

In [ ]:
# 2D scatter — VIX vs Theta colored by outcome
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(good_df_vix['VIX'], good_df_vix['Theta'], 
           alpha=0.15, s=10, color='green', label='Good trade')
ax.scatter(trap_df_vix['VIX'], trap_df_vix['Theta'],
           alpha=0.15, s=10, color='red', label='Trap')
ax.axvline(25, color='black', linestyle='--', alpha=0.3)
ax.axhline(5, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('VIX', fontsize=12)
ax.set_ylabel('$\\theta$ (mean-reversion speed)', fontsize=12)
ax.set_title('Trade Outcome in VIX-Theta Space\n'
             'Bottom-right = high VIX + slow reversion = danger zone',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('This plot is the core motivation for the project.')
print('If good trades and traps were perfectly mixed, no model could help.')
print('Separation in this space = the DQN has something to learn.')

---
## Summary

| Data | Source | Shape | Purpose |
|---|---|---|---|
| Daily prices | yfinance | 3,522 days x 47 tickers | Compute spreads |
| 10Y Yield, HY Spread | FRED | 3,697 days x 2 | Macro regime context |
| VIX | yfinance | 3,522 days | Volatility regime context |
| Spreads | Derived (rolling OLS) | 3,270 days x 24 pairs | Trading signal |
| OU params | Derived (AR(1) on 60d windows) | ~3,200 x 19 pairs | Mean-reversion dynamics |
| Feature matrix | Derived (14 features) | ~41,200 total obs | DQN state input |
| PCA-compressed | Derived (95% variance) | 11 components + 3 position = 14 dim | Actual DQN input |

**Key insight from this EDA:** Structural traps are predictable — they cluster in high-VIX, 
low-theta, high-credit-spread regimes. This is why an adaptive execution layer (our DQN) 
can add value over static thresholds.